# Text Preprocessing with NLTK

Tokenization, stemming vs. lemmatization, stop word removal, POS tagging, and named entity recognition on a short real paragraph of text.

In [1]:
import nltk

# Idempotent downloads (safe to re-run)
for pkg in ["punkt", "punkt_tab", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng",
            "wordnet", "omw-1.4", "stopwords", "maxent_ne_chunker", "maxent_ne_chunker_tab", "words"]:
    nltk.download(pkg, quiet=True)
print("NLTK resources ready.")

NLTK resources ready.


## Sample text

A short paragraph (2-3 sentences) that we'll run through the full preprocessing pipeline.

In [2]:
text = (
    "Barack Obama was born in Hawaii and served as the 44th President of the United States. "
    "He studied at Harvard University before moving to Chicago in 1985. "
    "The Nobel Committee awarded him the Nobel Peace Prize in 2009."
)
print(text)

Barack Obama was born in Hawaii and served as the 44th President of the United States. He studied at Harvard University before moving to Chicago in 1985. The Nobel Committee awarded him the Nobel Peace Prize in 2009.


## Tokenization

Split the paragraph into sentences with `sent_tokenize`, then each sentence into words with `word_tokenize`.

In [3]:
from nltk.tokenize import sent_tokenize, word_tokenize

sentences = sent_tokenize(text)
print(f"Number of sentences: {len(sentences)}\n")
for i, s in enumerate(sentences, 1):
    print(f"{i}: {s}")

Number of sentences: 3

1: Barack Obama was born in Hawaii and served as the 44th President of the United States.
2: He studied at Harvard University before moving to Chicago in 1985.
3: The Nobel Committee awarded him the Nobel Peace Prize in 2009.


In [4]:
words = word_tokenize(text)
print(f"Number of tokens: {len(words)}")
print(words)

Number of tokens: 41
['Barack', 'Obama', 'was', 'born', 'in', 'Hawaii', 'and', 'served', 'as', 'the', '44th', 'President', 'of', 'the', 'United', 'States', '.', 'He', 'studied', 'at', 'Harvard', 'University', 'before', 'moving', 'to', 'Chicago', 'in', '1985', '.', 'The', 'Nobel', 'Committee', 'awarded', 'him', 'the', 'Nobel', 'Peace', 'Prize', 'in', '2009', '.']


Basic terminology check: vocabulary is the set of *unique* tokens (case-folded).

In [5]:
vocabulary = sorted(set(w.lower() for w in words if w.isalpha()))
print(f"Vocabulary size: {len(vocabulary)}")
print(vocabulary)

Vocabulary size: 29
['and', 'as', 'at', 'awarded', 'barack', 'before', 'born', 'chicago', 'committee', 'harvard', 'hawaii', 'he', 'him', 'in', 'moving', 'nobel', 'obama', 'of', 'peace', 'president', 'prize', 'served', 'states', 'studied', 'the', 'to', 'united', 'university', 'was']


## Stemming vs. Lemmatization

Compare `PorterStemmer` against `WordNetLemmatizer` side-by-side. Lemmatization is run in noun mode (default) and also verb mode to show POS-awareness.

In [6]:
from nltk.stem import PorterStemmer, SnowballStemmer, WordNetLemmatizer
import pandas as pd

porter = PorterStemmer()
snowball = SnowballStemmer("english")
lemmatizer = WordNetLemmatizer()

sample_words = ["studies", "studying", "universal", "university", "better", "running",
                "presidency", "awarded", "nations", "flies"]

comparison = pd.DataFrame({
    "word": sample_words,
    "porter_stem": [porter.stem(w) for w in sample_words],
    "snowball_stem": [snowball.stem(w) for w in sample_words],
    "lemma_as_noun": [lemmatizer.lemmatize(w, pos="n") for w in sample_words],
    "lemma_as_verb": [lemmatizer.lemmatize(w, pos="v") for w in sample_words],
})
comparison

,word,porter_stem,snowball_stem,lemma_as_noun,lemma_as_verb
0,studies,studi,studi,study,study
1,studying,studi,studi,studying,study
2,universal,univers,univers,universal,universal
3,university,univers,univers,university,university
4,better,better,better,better,better
5,running,run,run,running,run
6,presidency,presid,presid,presidency,presidency
7,awarded,award,award,awarded,award
8,nations,nation,nation,nation,nations
9,flies,fli,fli,fly,fly


Notice the over-stemming pitfall: `universal` and `university` both stem to `univers` under Porter, even though they are semantically unrelated — a classic collision that lemmatization avoids.

## From-scratch: a tiny hand-rolled stemmer

To see that stemming really is just rule-based pattern matching (not something magical `PorterStemmer` alone knows how to do), we build a toy stemmer with just **four ordered rules** — strip `-ing`, `-ed`, `-s`, `-ly` if the remaining stem is at least 3 characters long — and compare its output word-by-word against `PorterStemmer` on the same sample words.

In [7]:
def toy_stemmer(word, min_stem_len=3):
    """A 4-rule hand-rolled suffix-stripping stemmer.

    Rules are checked in order; the first matching suffix is stripped,
    but only if what remains is at least `min_stem_len` characters long
    (avoids mangling very short words).
    """
    rules = ["ing", "ed", "ly", "s"]
    for suffix in rules:
        if word.endswith(suffix) and len(word) - len(suffix) >= min_stem_len:
            return word[: -len(suffix)]
    return word


toy_words = ["jumps", "jumping", "jumped", "quickly", "studies", "studying",
             "universal", "university", "flies", "running", "awarded", "nations"]

toy_vs_porter = pd.DataFrame({
    "word": toy_words,
    "toy_stem": [toy_stemmer(w) for w in toy_words],
    "porter_stem": [porter.stem(w) for w in toy_words],
})
toy_vs_porter["agree"] = toy_vs_porter["toy_stem"] == toy_vs_porter["porter_stem"]
toy_vs_porter

,word,toy_stem,porter_stem,agree
0,jumps,jump,jump,True
1,jumping,jump,jump,True
2,jumped,jump,jump,True
3,quickly,quick,quickli,False
4,studies,studie,studi,False
5,studying,study,studi,False
6,universal,universal,univers,False
7,university,university,univers,False
8,flies,flie,fli,False
9,running,runn,run,False


**Agreement** (4/12 words): on simple regular inflections — `jumps`/`jumping`/`jumped` → `jump`, and `awarded` → `award`, `nations` → `nation` — the toy 4-rule stemmer reproduces Porter's output exactly, because a flat suffix strip is all that's needed.

**Disagreement** (8/12 words) exposes exactly what the toy stemmer is missing relative to Porter's real 5-phase rule table:
- `quickly` → toy gives `quick` (correct-looking), but Porter gives `quickli` — Porter's `-ly` handling interacts with its other phases differently than a simple strip.
- `studies`/`flies` → toy gives `studie`/`flie` (not real words), while Porter gives `studi`/`fli` — Porter applies a `IES → I` rule instead of a blind `-s` strip, which the toy stemmer has no equivalent for.
- `universal` → toy leaves it unchanged (no rule matches: it doesn't end in `ing`/`ed`/`ly`/`s`), while Porter's derivational-suffix phases strip it down to `univers` — this is exactly the over-stemming pitfall from the comparison table above, and the toy stemmer's narrower rule set simply never triggers it.
- `running` → toy gives `runn` (a doubled consonant left dangling), Porter gives `run` — Porter includes a doubled-consonant cleanup step after suffix stripping that the toy stemmer skips entirely.

This confirms the point directly: stemming is pattern matching over a rule table, not a magic semantic process. A 4-rule toy version gets the simple, regular cases exactly right and produces the *same kind* of mechanical, sometimes-non-word output as Porter — it just has fewer, less careful rules, so it either under-reduces words that need more sophisticated handling (`universal`) or produces cruder non-words than Porter's more careful phases would (`studie` vs. `studi`).

## Stop Word Removal

In [8]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))
filtered_words = [w for w in words if w.lower() not in stop_words and w.isalpha()]

print(f"Before removal: {len([w for w in words if w.isalpha()])} alphabetic tokens")
print(f"After removal:  {len(filtered_words)} tokens")
print(filtered_words)

Before removal: 35 alphabetic tokens
After removal:  19 tokens
['Barack', 'Obama', 'born', 'Hawaii', 'served', 'President', 'United', 'States', 'studied', 'Harvard', 'University', 'moving', 'Chicago', 'Nobel', 'Committee', 'awarded', 'Nobel', 'Peace', 'Prize']


## Part-of-Speech (POS) Tagging

Tag every token in the original sentence with its Penn Treebank POS tag.

In [9]:
pos_tags = nltk.pos_tag(words)
pd.DataFrame(pos_tags, columns=["token", "pos_tag"])

,token,pos_tag
0,Barack,NNP
1,Obama,NNP
2,was,VBD
3,born,VBN
4,in,IN
5,Hawaii,NNP
6,and,CC
7,served,VBD
8,as,IN
9,the,DT


## Named Entity Recognition (NER)

Run `ne_chunk` over the POS-tagged tokens to identify entity spans (PERSON, ORGANIZATION, GPE, ...).

In [10]:
ner_tree = nltk.ne_chunk(pos_tags)
print(ner_tree)

(S
  (PERSON Barack/NNP)
  (PERSON Obama/NNP)
  was/VBD
  born/VBN
  in/IN
  (GPE Hawaii/NNP)
  and/CC
  served/VBD
  as/IN
  the/DT
  44th/CD
  President/NNP
  of/IN
  the/DT
  (GPE United/NNP States/NNPS)
  ./.
  He/PRP
  studied/VBD
  at/IN
  (ORGANIZATION Harvard/NNP University/NNP)
  before/IN
  moving/VBG
  to/TO
  (GPE Chicago/NNP)
  in/IN
  1985/CD
  ./.
  The/DT
  (ORGANIZATION Nobel/NNP Committee/NNP)
  awarded/VBD
  him/PRP
  the/DT
  (ORGANIZATION Nobel/NNP Peace/NNP Prize/NNP)
  in/IN
  2009/CD
  ./.)


In [11]:
# Extract just the recognised named entities in a readable form
entities = []
for subtree in ner_tree:
    if hasattr(subtree, "label"):
        entity_text = " ".join(token for token, tag in subtree.leaves())
        entities.append((entity_text, subtree.label()))

pd.DataFrame(entities, columns=["entity", "type"])

,entity,type
0,Barack,PERSON
1,Obama,PERSON
2,Hawaii,GPE
3,United States,GPE
4,Harvard University,ORGANIZATION
5,Chicago,GPE
6,Nobel Committee,ORGANIZATION
7,Nobel Peace Prize,ORGANIZATION


The NER pass correctly recognises entities like *Barack Obama* (PERSON), *Hawaii* / *United States* (GPE), *Harvard University* (ORGANIZATION), and *Chicago* (GPE) directly from the raw sentence, demonstrating the full preprocessing pipeline: tokenize → POS-tag → chunk into named entities.